# 04 · GraphRAG, Hybrid và Đánh giá

Ghép Basic RAG với đồ thị bằng Text2Cypher và router, đo bằng Recall@k, MRR, RAGAS, rồi so ba hệ thống trên cùng bộ benchmark.

**Đầu ra:** `results/results.parquet`, `src/hybrid.py` (`router`, `ask_hybrid`), `src/eval.py`.

**Lưu ý:** P4-03 có hai lớp chặn: regex và giao dịch chỉ đọc của Neo4j. P4-13: nếu Hybrid không thắng ở loại câu hỏi nào, giữ nguyên kết quả và giải thích. Kết luận trên slide 30 phải đến từ số đo.

In [ ]:
%run ./00_setup.ipynb

### P4-01 · Cả ba hệ thống chạy trên cùng dữ liệu và cùng bộ câu hỏi
**ẨN**

Mục tiêu: nạp benchmark/questions.csv, ask_rag của Người 2, kết nối đồ thị của Người 3; thêm baseline LLM only (không ngữ cảnh).

In [ ]:
# P4-01 · ẨN
df = pd.read_parquet(CLEAN_PATH)
questions_raw = pd.read_csv(QUESTIONS_PATH, keep_default_na=False)
graph.driver()
assert qdrant_store.count() > 0, "chạy notebook 01 trước để nạp Qdrant"
# llm_only: baseline không truy xuất, để đo RAG và đồ thị thêm được gì so với LLM trần
SYSTEMS = {"llm_only": rag.ask_llm_only, "basic_rag": ask_rag, "graph": ask_graph, "hybrid": ask_hybrid}

### P4-02 · Đáp án chuẩn tính bằng pandas, không gõ tay
**CACHE**

Mục tiêu: tính đáp án chuẩn bằng pandas groupby cho câu tổng hợp và đa chặng, ghi vào cột gt_value.

In [ ]:
# P4-02 · CACHE
questions = cached("p4_02_questions.pkl", lambda: ev.fill_ground_truth(questions_raw, df))
questions.assign(gt_ids=questions.gt_ids.map(json.dumps)).to_csv(QUESTIONS_PATH, index=False)
print(questions.type.value_counts().to_dict())
show(questions[questions.type.isin(ev.VALUE_TYPES)], ["id", "type", "question", "gt_value"], rows=4)

### P4-03 · Text2Cypher sinh truy vấn chỉ đọc, lệnh ghi bị chặn hai lớp
**LIVE**

Mục tiêu: prompt chứa schema, sinh Cypher, bộ chặn chỉ đọc, thực thi, thử lại khi lỗi.

In [ ]:
# P4-03 · LIVE
from neo4j.exceptions import Neo4jError
assert graph.is_read_only("MATCH (l:Listing) RETURN count(l) AS so_tin")
assert not graph.is_read_only("MATCH (l:Listing) DETACH DELETE l")

t2c = graph.text2cypher("Ở Hà Nội, quận nào có giá/m² trung bình cao nhất (chỉ tính quận có ít nhất 20 tin)?")
print(t2c["cypher"], f"\n(số lần thử: {len(t2c['attempts'])})")
print(t2c["rows"].head(5).to_string(index=False))
try:  # lớp 2: giao dịch chỉ đọc của Neo4j chặn cả câu lọt qua regex (gọi thẳng run_cypher)
    run_cypher("MATCH (l:Listing {listing_id: -1}) SET l.x = 1")
except Neo4jError as e:
    print("\nví dụ bị chặn ở cơ sở dữ liệu:", e.code)

### P4-04 · Local retrieval mở rộng láng giềng quanh một thực thể
**CACHE**

Mục tiêu: mở rộng láng giềng quanh 1 phường, vẽ đồ thị con.

In [ ]:
# P4-04 · CACHE
key = df.dropna(subset=["ward"]).groupby(["province", "district", "ward"]).size().idxmax()
G = graph.neighborhood("|".join(key), limit=10)
print(f"{key[2]}, {key[1]}: {G.number_of_nodes()} nút, {G.number_of_edges()} cạnh")
graph.draw_graph(G, title=f"Láng giềng của {key[2]} ({key[1]}, {key[0]})");

### P4-05 · Global search chỉ giải thích, không chạy vì dữ liệu có cấu trúc
**ẨN**

Mục tiêu: giải thích khái niệm tóm tắt cộng đồng; câu tổng hợp đã có Cypher chính xác hơn.

In [ ]:
# P4-05 · ẨN
# Global search của GraphRAG: tóm tắt từng cộng đồng bằng LLM rồi map-reduce qua các bản tóm tắt.
# Ở đây cột giá, diện tích, địa danh đã có cấu trúc, nên câu tổng hợp trả lời chính xác bằng Cypher
# (P3-10) thay vì tóm tắt LLM. Cộng đồng phường ở P3-12 dùng được làm đầu vào nếu muốn thử sau.
GLOBAL_SEARCH = False

### P4-06 · Router chọn đúng nguồn cho [ĐIỀN SAU]/8 câu
**LIVE**

Mục tiêu: phân loại câu hỏi thành basic_rag, graph, hybrid bằng vài ví dụ mẫu; thử trên 8 câu.

In [ ]:
# P4-06 · LIVE
eight = questions[questions.type != ev.UNANSWERABLE].groupby("type").head(2)
routes = [router(q) for q in eight.question]
table = eight[["id", "question", "route"]].rename(columns={"route": "nhãn đúng"})
table["dự đoán"] = [r["route"] for r in routes]
print(f"đúng {(table['nhãn đúng'] == table['dự đoán']).sum()}/{len(table)}")
show(table, width=70)

### P4-07 · ask_hybrid gộp ngữ cảnh hai nguồn và có đường lui về Basic RAG
**ẨN**

Mục tiêu: ask_hybrid: gộp ngữ cảnh từ hai nguồn, lui về Basic RAG khi đồ thị lỗi hoặc rỗng (src/hybrid.py).

In [ ]:
# P4-07 · ẨN
_h = ask_hybrid("Nhà ở Cầu Giấy dưới 10 tỷ có thang máy.")
assert {"answer", "sources", "route", "fallback"} <= set(_h)

### P4-08 · Recall@5 = 0,67 và MRR = 0,61 ở toy; số thật [ĐIỀN SAU]
**LIVE**

Mục tiêu: hàm recall_at_k và mrr: chạy toy trước, sau đó chạy phần truy xuất trên benchmark.

In [ ]:
# P4-08 · LIVE
retrieved, relevant = ["d4", "d2", "d1", "d9", "d3"], {"d1", "d4", "d7"}
runs = [(["a", "x", "y"], {"a"}), (["x", "y", "b"], {"b"}), (["x", "c", "y"], {"c"})]
assert abs(ev.recall_at_k(retrieved, relevant, 5) - 2 / 3) < 1e-9
assert abs(ev.mrr(runs) - (1 + 1 / 3 + 1 / 2) / 3) < 1e-9
print(f"toy: Recall@5 = {ev.recall_at_k(retrieved, relevant, 5):.2f}, MRR = {ev.mrr(runs):.2f}")

rq = questions[questions.type.isin(ev.RETRIEVAL_TYPES)]
methods = {"dense": lambda q: search_dense(q, 5), "BM25": lambda q: search_bm25(q, 5),
           "hybrid + lọc": lambda q: rag.retrieve(q)["hits"]}
ret = cached("p4_08_retrieval.pkl", lambda: {m: [f(q).listing_id.tolist() for q in rq.question] for m, f in methods.items()})
pd.DataFrame({m: {"Recall@5": np.mean([ev.recall_at_k(r, set(g), 5, capped=True) for r, g in zip(ids, rq.gt_ids)]),
                  "MRR": ev.mrr([(r, set(g)) for r, g in zip(ids, rq.gt_ids)])} for m, ids in ret.items()}).round(2)

### P4-09 · Faithfulness toy = 3/4 = 0,75; RAGAS trên tập con [ĐIỀN SAU]
**CACHE**

Mục tiêu: Faithfulness, Answer Relevancy, Context Precision trên một tập con; kèm hàm Faithfulness thủ công cho toy.

In [ ]:
# P4-09 · CACHE
assert ev.faithfulness([True, True, True, False]) == 0.75

def ragas_subset():
    rows, sub = [], questions[questions.type != ev.UNANSWERABLE].groupby("type").head(1)
    for q in sub.itertuples():
        reference = q.gt_value or f"các tin {list(q.gt_ids)[:5]}"
        for name in ("basic_rag", "hybrid"):
            r = SYSTEMS[name](q.question)
            rows.append({"id": q.id, "system": name,
                         "faithfulness": ev.faithfulness(ev.claims_supported(r["answer"], r["contexts"])),
                         "answer_relevancy": ev.answer_relevancy(q.question, r["answer"]),
                         "context_precision": ev.context_precision(q.question, r["contexts"], reference)})
    return pd.DataFrame(rows)

ragas = cached("p4_09_ragas.parquet", ragas_subset)
ragas.groupby("system")[["faithfulness", "answer_relevancy", "context_precision"]].mean().round(2)

### P4-10 · Benchmark chạy đủ 4 hệ thống trên toàn bộ câu hỏi
**CACHE**

Mục tiêu: chạy toàn bộ benchmark (gồm 4 câu cố ý không có đáp án) qua 4 hệ thống, ghi câu trả lời, độ trễ, số token vào results.parquet.

In [ ]:
# P4-10 · CACHE
from src.llm import fresh

def run_all():
    with fresh(), Timer() as t:  # gọi API thật để độ trễ và token phản ánh đúng chi phí
        runs = ev.run_benchmark(questions, SYSTEMS)
    print(f"chạy mới trong {t.seconds:.0f} s")
    return ev.score(runs, questions)

results = cached(RESULTS_PATH, run_all)
print(f"{len(results)} dòng = {questions.shape[0]} câu × {results.system.nunique()} hệ thống | "
      f"tổng độ trễ {results.latency.sum():.0f} s | lỗi: {results.error.notna().sum()}")

### P4-11 · Hybrid thắng ở [ĐIỀN SAU], Basic RAG thua rõ ở câu tổng hợp
**LIVE**

Mục tiêu: bảng độ chính xác theo loại câu hỏi và hệ thống (có LLM only và tỷ lệ từ chối đúng ở câu không có đáp án); điểm trung bình thang 0–2 tính cả câu đúng một phần; biểu đồ cột nhóm.

In [ ]:
# P4-11 · LIVE
acc = ev.accuracy_table(results)
ax = acc.plot.bar(figsize=(10, 4), rot=0, color=["#9D9D9D", "#4C78A8", "#F58518", "#54A24B"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("tỷ lệ đúng")
best = acc.idxmax(axis=1)
ax.set_title("Hệ thống tốt nhất theo loại: " + ", ".join(f"{t}: {s}" for t, s in best.items()))
print("điểm trung bình (0–2):")
print(ev.accuracy_table(results, "points").to_string())
acc

### P4-12 · Graph và Hybrid tốn thêm [ĐIỀN SAU] token và [ĐIỀN SAU] giây mỗi câu
**CACHE**

Mục tiêu: bảng chi phí và độ trễ trung bình theo hệ thống.

In [ ]:
# P4-12 · CACHE
cost = cached("p4_12_cost.parquet", lambda: ev.cost_table(results).reset_index())
cost.rename(columns={"latency_s": "độ trễ TB (s)", "input_tokens": "token vào TB",
                     "output_tokens": "token ra TB", "usd_per_question": "USD/câu"})

### P4-13 · Lỗi chủ yếu đến từ router nhầm và Cypher sai [ĐIỀN SAU]
**CACHE**

Mục tiêu: phân tích lỗi: 2–3 trường hợp sai (router nhầm, Cypher sai).

In [ ]:
# P4-13 · CACHE
errors = cached("p4_13_errors.parquet", lambda: ev.error_cases(results, n=3))
print(results[~results.correct].groupby("system").size().rename("số câu sai").to_string())
show(errors, width=60)